In [7]:
# ============================================================
# STYLE EXTRACTION PIPELINE — SETUP
# Notebook location : /notebooks
# Input report      : /notebooks/gen_data/style/
# Outputs           : /notebooks/gen_data/style/style_system/
# ============================================================

import os
import re
import json
import time
import random
import urllib.request
import urllib.error
import http.client
from pathlib import Path

import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError:
    raise ImportError("Install PyMuPDF first: pip install pymupdf")

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    raise ImportError("Install python-dotenv first: pip install python-dotenv")


# ------------------------------------------------------------
# 1. Resolve notebook/project paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

# Robust handling:
# - if the notebook is launched from /notebooks, use that
# - if launched from project root, use project_root/notebooks
# - otherwise, fall back to current directory
if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

STYLE_DATA_DIR = NOTEBOOK_DIR / "gen_data" / "style"

if not STYLE_DATA_DIR.exists():
    raise FileNotFoundError(
        "Style data directory not found.\n"
        f"Expected path: {STYLE_DATA_DIR}\n"
        "Create this folder and place the reference report inside it."
    )


# ------------------------------------------------------------
# 2. Input reference report
# ------------------------------------------------------------

DEFAULT_REFERENCE_REPORT_NAME = "emirates_nbd_group_2024_ifrs_s1_s2.pdf"

reference_report_env = os.getenv("REFERENCE_REPORT_PDF")

if reference_report_env:
    candidate = Path(reference_report_env)

    if candidate.is_absolute():
        REFERENCE_REPORT_PDF = candidate
    else:
        # First try relative to the style data folder
        if (STYLE_DATA_DIR / candidate).exists():
            REFERENCE_REPORT_PDF = STYLE_DATA_DIR / candidate
        # Then try relative to notebook directory
        elif (NOTEBOOK_DIR / candidate).exists():
            REFERENCE_REPORT_PDF = NOTEBOOK_DIR / candidate
        # Then try relative to current working directory
        else:
            REFERENCE_REPORT_PDF = CURRENT_DIR / candidate
else:
    REFERENCE_REPORT_PDF = STYLE_DATA_DIR / DEFAULT_REFERENCE_REPORT_NAME

REFERENCE_REPORT_PDF = REFERENCE_REPORT_PDF.resolve()

if not REFERENCE_REPORT_PDF.exists():
    raise FileNotFoundError(
        f"Reference report not found: {REFERENCE_REPORT_PDF}\n\n"
        "Expected default location:\n"
        f"{STYLE_DATA_DIR / DEFAULT_REFERENCE_REPORT_NAME}\n\n"
        "Either place the PDF there, or set REFERENCE_REPORT_PDF in your .env."
    )


# ------------------------------------------------------------
# 3. Output folders
# ------------------------------------------------------------

# Outputs are placed under the same directory as the input report.
STYLE_OUTPUT_DIR = REFERENCE_REPORT_PDF.parent / "style_system"

SECTION_STYLE_DIR = STYLE_OUTPUT_DIR / "section_style_guides"
SECTION_BLUEPRINT_DIR = STYLE_OUTPUT_DIR / "section_blueprints"
TABLE_PATTERN_DIR = STYLE_OUTPUT_DIR / "table_patterns"
LANGUAGE_RULES_DIR = STYLE_OUTPUT_DIR / "language_rules"
INTERMEDIATE_DIR = STYLE_OUTPUT_DIR / "_intermediate"

for folder in [
    STYLE_OUTPUT_DIR,
    SECTION_STYLE_DIR,
    SECTION_BLUEPRINT_DIR,
    TABLE_PATTERN_DIR,
    LANGUAGE_RULES_DIR,
    INTERMEDIATE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 4. Target sections
# ------------------------------------------------------------

TARGET_SECTIONS = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

SECTION_SLUGS = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("Current working directory:", CURRENT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Style data directory:", STYLE_DATA_DIR)
print("Reference report:", REFERENCE_REPORT_PDF)
print("Style output folder:", STYLE_OUTPUT_DIR)

Current working directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Notebook directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Style data directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style
Reference report: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\emirates_nbd_group_2024_ifrs_s1_s2.pdf
Style output folder: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system


In [8]:
# ============================================================
# PDF TEXT EXTRACTION
# ============================================================

def clean_pdf_text(text: str) -> str:
    if not text:
        return ""

    # Normalize strange whitespace
    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("", " ")
    text = text.replace("￾", "-")

    # Fix common PDF extraction issue where words are glued less aggressively.
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_pdf_pages(pdf_path: Path) -> pd.DataFrame:
    doc = fitz.open(str(pdf_path))
    rows = []

    for page_index, page in enumerate(doc):
        page_number = page_index + 1
        text = clean_pdf_text(page.get_text("text"))

        rows.append({
            "page": page_number,
            "text": text,
            "char_count": len(text),
        })

    return pd.DataFrame(rows)


pages_df = extract_pdf_pages(REFERENCE_REPORT_PDF)

print("Pages extracted:", len(pages_df))
display(pages_df.head())

Pages extracted: 69


,page,text,char_count
0,1,Emirates NBD Group \n2024 IFRS S1 and S2 \nReport,47
1,2,2\nEmirates NBD Group 2024 IFRS S1 and S2 Report,47
2,3,Contents\nGeneral Requirements \n 4\n1.1 Und...,2158
3,4,01\nGeneral \nRequirements,24
4,5,General Requirements\nGeneral Requirements\nTh...,3950


In [9]:
# ============================================================
# SECTION SEGMENTATION
# ============================================================

# For the reference report, these section boundaries are visible in the official contents page.
# This is acceptable because style extraction is report-specific.
# We are not using this for IFRS compliance extraction.

REFERENCE_SECTION_STARTS = {
    "General Requirements": 4,
    "Governance": 8,
    "Strategy": 19,
    "Risk Management": 47,
    "Metrics and Targets": 55,
    "Appendix": 64,
}


def build_section_ranges(section_starts: dict) -> dict:
    ordered = sorted(section_starts.items(), key=lambda x: x[1])
    ranges = {}

    for i, (section, start_page) in enumerate(ordered):
        if section == "Appendix":
            continue

        next_start = ordered[i + 1][1] if i + 1 < len(ordered) else int(pages_df["page"].max()) + 1
        end_page = next_start - 1

        ranges[section] = {
            "start_page": start_page,
            "end_page": end_page,
        }

    return ranges


section_ranges = build_section_ranges(REFERENCE_SECTION_STARTS)

print(json.dumps(section_ranges, indent=2))


def get_section_text(section_name: str) -> str:
    start = section_ranges[section_name]["start_page"]
    end = section_ranges[section_name]["end_page"]

    subset = pages_df[
        (pages_df["page"] >= start) &
        (pages_df["page"] <= end)
    ].copy()

    joined = "\n\n".join(
        f"[PAGE {row.page}]\n{row.text}"
        for _, row in subset.iterrows()
        if str(row.text).strip()
    )

    return clean_pdf_text(joined)


section_texts = {
    section: get_section_text(section)
    for section in TARGET_SECTIONS
}

for section, text in section_texts.items():
    print(section, "chars:", len(text))

{
  "General Requirements": {
    "start_page": 4,
    "end_page": 7
  },
  "Governance": {
    "start_page": 8,
    "end_page": 18
  },
  "Strategy": {
    "start_page": 19,
    "end_page": 46
  },
  "Risk Management": {
    "start_page": 47,
    "end_page": 54
  },
  "Metrics and Targets": {
    "start_page": 55,
    "end_page": 63
  }
}
General Requirements chars: 11089
Governance chars: 34893
Strategy chars: 82610
Risk Management chars: 24039
Metrics and Targets chars: 27044


In [10]:
# ============================================================
# AZURE OPENAI REST HELPER
# ============================================================

env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print("Loaded .env from:", env_path)
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

AZURE_OPENAI_STYLE_URL = _clean_url(
    os.getenv("AZURE_OPENAI_STYLE_URL")
    or os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
    or os.getenv("AZURE_OPENAI_EXTRACTOR_URL")
    or os.getenv("AZURE_OPENAI_JUDGE_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)

STYLE_MAX_OUTPUT_TOKENS = int(os.getenv("STYLE_MAX_OUTPUT_TOKENS", "8000"))
STYLE_CHUNK_MAX_CHARS = int(os.getenv("STYLE_CHUNK_MAX_CHARS", "10000"))
STYLE_INTER_REQUEST_DELAY_SECONDS = float(os.getenv("STYLE_INTER_REQUEST_DELAY_SECONDS", "1.0"))


def validate_style_azure_config():
    missing = []

    if not AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")

    if not AZURE_OPENAI_STYLE_URL:
        missing.append(
            "AZURE_OPENAI_STYLE_URL or AZURE_OPENAI_GPT52_DEPLOYMENT_URL or AZURE_OPENAI_JUDGE_URL"
        )

    if missing:
        raise ValueError(
            "Missing Azure style extraction configuration: "
            + ", ".join(missing)
        )

    if not AZURE_OPENAI_STYLE_URL.startswith("https://"):
        raise ValueError(
            "Azure URL must be a full HTTPS endpoint. "
            f"Current value: {AZURE_OPENAI_STYLE_URL!r}"
        )

    print("Azure style extraction config loaded.")
    print("Endpoint:", AZURE_OPENAI_STYLE_URL[:120] + "...")
    print("API key loaded:", bool(AZURE_OPENAI_API_KEY))


validate_style_azure_config()


def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except Exception as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned empty content:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _extract_json_object(text: str) -> str:
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def azure_chat_json(
    *,
    system_prompt: str,
    user_prompt: str,
    request_label: str,
    max_output_tokens: int = STYLE_MAX_OUTPUT_TOKENS,
    timeout: int = 240,
    max_attempts: int = 5,
) -> dict:
    token_fields = ["max_completion_tokens", "max_tokens"]
    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            token_field: max_output_tokens,
            "temperature": 0.0,
            "response_format": {"type": "json_object"},
        }

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                AZURE_OPENAI_STYLE_URL,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": AZURE_OPENAI_API_KEY,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    data = json.loads(resp.read().decode("utf-8"))

                content = _extract_message_content(data)
                candidate = _extract_json_object(content)
                return json.loads(candidate)

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Token field used: {token_field}\n"
                    f"Response preview: {body[:2000]}"
                )

                transient = exc.code in {429, 500, 502, 503, 504}
                compatibility = token_field == "max_completion_tokens" and exc.code in {400, 422, 500}

                if transient and attempt < max_attempts:
                    retry_after = exc.headers.get("Retry-After")
                    try:
                        wait = float(retry_after) if retry_after else min(2 ** attempt, 30)
                    except ValueError:
                        wait = min(2 ** attempt, 30)

                    print(f"{request_label}: HTTP {exc.code}; retrying in {wait:.1f}s...")
                    time.sleep(wait)
                    continue

                if compatibility:
                    print(f"{request_label}: retrying with max_tokens instead of max_completion_tokens.")
                    break

                raise last_error from exc

            except (
                urllib.error.URLError,
                ConnectionResetError,
                TimeoutError,
                OSError,
                http.client.RemoteDisconnected,
                json.JSONDecodeError,
            ) as exc:
                last_error = RuntimeError(
                    f"{request_label} failed on attempt {attempt}/{max_attempts}: {repr(exc)}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** attempt + random.random(), 30)
                    print(f"{request_label}: transient error; retrying in {wait:.1f}s...")
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(f"{request_label} failed.")

Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Azure style extraction config loaded.
Endpoint: https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.2/chat/completions...
API key loaded: True


In [13]:
# ============================================================
# LLM STYLE NOTE EXTRACTION
# ============================================================

STYLE_CHUNK_SYSTEM_PROMPT = """
You are a senior sustainability reporting style analyst.

You extract reusable style, structure, formatting, and disclosure presentation patterns from a reference IFRS S1/S2 sustainability report.

Important:
- Do not copy sentences from the reference report.
- Do not extract company-specific facts, metrics, names, amounts, achievements, targets, people, committees, or claims as reusable content.
- Focus only on abstract writing style and structure.
- The output will be used to guide report generation for a different company.
- Return valid JSON only.
""".strip()


def build_style_chunk_prompt(section_name: str, chunk_index: int, chunk_text: str) -> str:
    return f"""
Analyze this excerpt from the reference report section: {section_name}.

Extract only reusable style and structure patterns.

Return JSON with this schema:
{{
  "section_name": "{section_name}",
  "chunk_index": {chunk_index},
  "tone_patterns": [],
  "paragraph_patterns": [],
  "heading_patterns": [],
  "table_patterns": [],
  "figure_or_diagram_patterns": [],
  "disclosure_language_patterns": [],
  "evidence_presentation_patterns": [],
  "section_specific_observations": [],
  "things_to_avoid_copying": [],
  "content_specific_items_detected_and_excluded": []
}}

Rules:
- Do not quote the reference report.
- Do not include Emirates NBD-specific names, facts, amounts, targets, awards, committees, people, or numbers as style rules.
- Do not include any copied sentence.
- Use abstract descriptions only.

REFERENCE EXCERPT:
{chunk_text}
""".strip()


style_chunk_notes = {}

for section_name, chunks in section_chunks.items():
    style_chunk_notes[section_name] = []

    for idx, chunk in enumerate(chunks, start=1):
        print(f"Extracting style notes: {section_name} chunk {idx}/{len(chunks)}")

        result = azure_chat_json(
            system_prompt=STYLE_CHUNK_SYSTEM_PROMPT,
            user_prompt=build_style_chunk_prompt(section_name, idx, chunk),
            request_label=f"Style notes {section_name} {idx}",
        )

        style_chunk_notes[section_name].append(result)

        time.sleep(STYLE_INTER_REQUEST_DELAY_SECONDS)


# Save raw notes
raw_notes_path = INTERMEDIATE_DIR / "style_chunk_notes.json"

with open(raw_notes_path, "w", encoding="utf-8") as f:
    json.dump(style_chunk_notes, f, ensure_ascii=False, indent=2)

print("Saved:", raw_notes_path)

Extracting style notes: General Requirements chunk 1/2
Style notes General Requirements 1: HTTP 502; retrying in 2.0s...
Extracting style notes: General Requirements chunk 2/2
Extracting style notes: Governance chunk 1/5
Style notes Governance 1: transient error; retrying in 2.6s...
Style notes Governance 1: transient error; retrying in 4.3s...
Extracting style notes: Governance chunk 2/5
Extracting style notes: Governance chunk 3/5
Extracting style notes: Governance chunk 4/5
Extracting style notes: Governance chunk 5/5
Extracting style notes: Strategy chunk 1/10
Extracting style notes: Strategy chunk 2/10
Extracting style notes: Strategy chunk 3/10
Extracting style notes: Strategy chunk 4/10
Extracting style notes: Strategy chunk 5/10
Extracting style notes: Strategy chunk 6/10
Extracting style notes: Strategy chunk 7/10
Extracting style notes: Strategy chunk 8/10
Extracting style notes: Strategy chunk 9/10
Extracting style notes: Strategy chunk 10/10
Extracting style notes: Risk Man

In [14]:
# ============================================================
# CHUNKING FOR STYLE EXTRACTION
# ============================================================

def split_text_into_chunks(text: str, max_chars: int = STYLE_CHUNK_MAX_CHARS) -> list[str]:
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        if len(current) + len(para) + 2 <= max_chars:
            current = current + "\n\n" + para if current else para
        else:
            if current:
                chunks.append(current)
            current = para

    if current:
        chunks.append(current)

    return chunks


section_chunks = {
    section: split_text_into_chunks(text)
    for section, text in section_texts.items()
}

for section, chunks in section_chunks.items():
    print(section, "chunks:", len(chunks), "chars:", sum(len(c) for c in chunks))

General Requirements chunks: 2 chars: 11079
Governance chunks: 5 chars: 34879
Strategy chunks: 10 chars: 82583
Risk Management chunks: 3 chars: 24033
Metrics and Targets chunks: 4 chars: 27035


In [15]:
# ============================================================
# CONSOLIDATE SECTION STYLE GUIDES
# ============================================================

SECTION_CONSOLIDATION_SYSTEM_PROMPT = """
You are a senior sustainability reporting architect.

You convert raw style notes into a clean, reusable section style guide and section blueprint.

Important:
- Remove all reference-company-specific content.
- Remove all numbers, amounts, achievements, names, and unique claims from the reference report.
- Keep only abstract, reusable style and structure rules.
- The output must be safe to use for generating a report for another company.
- Return valid JSON only.
""".strip()


def build_section_consolidation_prompt(section_name: str, notes: list[dict]) -> str:
    return f"""
Create a clean style guide and blueprint for the section: {section_name}.

Use the raw notes below, but remove all company-specific content.

Return JSON with this schema:
{{
  "section_name": "{section_name}",
  "section_style_guide": {{
    "purpose": "",
    "tone": "",
    "level_of_detail": "",
    "paragraph_style": "",
    "sentence_style": "",
    "preferred_disclosure_verbs": [],
    "preferred_evidence_style": "",
    "table_usage": "",
    "figure_usage": "",
    "how_to_discuss_missing_data": "",
    "what_to_avoid": []
  }},
  "section_blueprint": {{
    "recommended_subsections": [],
    "recommended_order": [],
    "recommended_tables": [],
    "recommended_figures_or_diagrams": [],
    "required_narrative_blocks": [],
    "optional_narrative_blocks": [],
    "output_structure_rules": []
  }},
  "section_specific_no_copying_rules": [],
  "quality_checks_for_generation": []
}}

Do not include:
- Emirates NBD-specific names
- person names
- committee names unique to the reference company
- amounts
- percentages
- achievements
- awards
- exact copied wording

RAW STYLE NOTES:
{json.dumps(notes, ensure_ascii=False, indent=2)}
""".strip()


section_style_outputs = {}

for section_name, notes in style_chunk_notes.items():
    print("Consolidating:", section_name)

    result = azure_chat_json(
        system_prompt=SECTION_CONSOLIDATION_SYSTEM_PROMPT,
        user_prompt=build_section_consolidation_prompt(section_name, notes),
        request_label=f"Consolidate {section_name}",
    )

    section_style_outputs[section_name] = result

    slug = SECTION_SLUGS[section_name]

    style_path = SECTION_STYLE_DIR / f"{slug}_style.json"
    blueprint_path = SECTION_BLUEPRINT_DIR / f"{slug}_blueprint.json"

    with open(style_path, "w", encoding="utf-8") as f:
        json.dump(result["section_style_guide"], f, ensure_ascii=False, indent=2)

    with open(blueprint_path, "w", encoding="utf-8") as f:
        json.dump(result["section_blueprint"], f, ensure_ascii=False, indent=2)

    print("Saved:", style_path)
    print("Saved:", blueprint_path)

    time.sleep(STYLE_INTER_REQUEST_DELAY_SECONDS)

Consolidating: General Requirements
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\section_style_guides\general_requirements_style.json
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\section_blueprints\general_requirements_blueprint.json
Consolidating: Governance
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\section_style_guides\governance_style.json
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\section_blueprints\governance_blueprint.json
Consolidating: Strategy
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\section_style_guides\strategy_style.json
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\section_blueprints\strategy_blueprint.json
Consolidating: Risk Management
Saved: C:\Users

In [16]:
# ============================================================
# GLOBAL STYLE GUIDE + TABLE PATTERNS
# ============================================================

GLOBAL_STYLE_SYSTEM_PROMPT = """
You are a senior sustainability reporting architect.

You create a global style guide from section-level style guides.

Important:
- Keep only abstract style rules.
- Do not include reference-company-specific facts, numbers, names, achievements, targets, people, or copied wording.
- Return valid JSON only.
""".strip()


global_prompt = f"""
Create the final global style system for an IFRS S1/S2-aligned sustainability report.

Return JSON with this schema:
{{
  "global_style_guide": {{
    "report_voice": "",
    "tone": "",
    "point_of_view": "",
    "paragraph_rules": [],
    "sentence_rules": [],
    "disclosure_language_rules": [],
    "evidence_and_traceability_rules": [],
    "missing_data_language_rules": [],
    "formatting_rules": [],
    "do_not_do": []
  }},
  "table_patterns": {{
    "general_table_rules": [],
    "governance_tables": [],
    "strategy_tables": [],
    "risk_management_tables": [],
    "metrics_and_targets_tables": [],
    "recommended_columns_by_table_type": {{}}
  }},
  "no_copying_rules": [
    "Do not copy sentences from the reference report.",
    "Do not reuse reference company facts, claims, amounts, targets, people, committee names or achievements.",
    "Use the reference only for abstract style, structure and formatting behavior."
  ],
  "style_compliance_checks": []
}}

SECTION STYLE OUTPUTS:
{json.dumps(section_style_outputs, ensure_ascii=False, indent=2)}
""".strip()


global_style_result = azure_chat_json(
    system_prompt=GLOBAL_STYLE_SYSTEM_PROMPT,
    user_prompt=global_prompt,
    request_label="Global style guide",
)


global_style_path = STYLE_OUTPUT_DIR / "global_style_guide.json"
table_patterns_path = TABLE_PATTERN_DIR / "table_patterns.json"
no_copying_path = LANGUAGE_RULES_DIR / "no_copying_rules.md"

with open(global_style_path, "w", encoding="utf-8") as f:
    json.dump(global_style_result["global_style_guide"], f, ensure_ascii=False, indent=2)

with open(table_patterns_path, "w", encoding="utf-8") as f:
    json.dump(global_style_result["table_patterns"], f, ensure_ascii=False, indent=2)

with open(no_copying_path, "w", encoding="utf-8") as f:
    f.write("# No-copying rules for report generation\n\n")
    for rule in global_style_result["no_copying_rules"]:
        f.write(f"- {rule}\n")

print("Saved:", global_style_path)
print("Saved:", table_patterns_path)
print("Saved:", no_copying_path)

Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\global_style_guide.json
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\table_patterns\table_patterns.json
Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\language_rules\no_copying_rules.md


In [17]:
# ============================================================
# STYLE ARTIFACT VALIDATION — NO COPYING / NO CONTENT LEAKAGE
# ============================================================

FORBIDDEN_REFERENCE_TERMS = [
    "Emirates NBD",
    "DenizBank",
    "Emirates Islamic",
    "Vijay Bains",
    "Manoj Chawla",
    "Patrick Sullivan",
    "BNRESGC",
    "BRC",
    "GRC",
    "AED",
    "USD",
    "Dubai",
    "UAE",
    "MENA",
    "MENAT",
    "Sustainable Finance Forum",
    "Board Nomination, Remuneration and Environmental Social Governance Committee",
]

AMOUNT_OR_METRIC_PATTERN = re.compile(
    r"(\b\d+(\.\d+)?\s?%|\bUSD\b|\bAED\b|\bbillion\b|\bmillion\b|\b\d{4}\b)",
    flags=re.IGNORECASE
)


def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="replace")


def validate_artifact_file(path: Path) -> dict:
    text = read_text_file(path)

    forbidden_hits = [
        term for term in FORBIDDEN_REFERENCE_TERMS
        if term.lower() in text.lower()
    ]

    metric_hits = AMOUNT_OR_METRIC_PATTERN.findall(text)

    return {
        "file": str(path),
        "forbidden_reference_terms": forbidden_hits,
        "amount_or_metric_like_patterns": [m[0] for m in metric_hits[:20]],
        "has_copying_risk": bool(forbidden_hits or metric_hits),
    }


artifact_paths = list(STYLE_OUTPUT_DIR.rglob("*.json")) + list(STYLE_OUTPUT_DIR.rglob("*.md"))

validation_rows = [
    validate_artifact_file(path)
    for path in artifact_paths
    if "_intermediate" not in str(path)
]

validation_df = pd.DataFrame(validation_rows)

validation_path = STYLE_OUTPUT_DIR / "style_artifact_validation.csv"
validation_df.to_csv(validation_path, index=False, encoding="utf-8-sig")

display(validation_df)

risky = validation_df[validation_df["has_copying_risk"] == True]

if len(risky):
    print("WARNING: Some style artifacts may contain reference-specific terms or numbers.")
    print("Review these files manually before using them in generation:")
    display(risky)
else:
    print("Validation passed: no obvious reference-specific leakage detected.")

forbidden_terms_path = LANGUAGE_RULES_DIR / "forbidden_reference_terms.json"

with open(forbidden_terms_path, "w", encoding="utf-8") as f:
    json.dump(FORBIDDEN_REFERENCE_TERMS, f, ensure_ascii=False, indent=2)

print("Saved validation:", validation_path)
print("Saved forbidden terms:", forbidden_terms_path)

,file,forbidden_reference_terms,amount_or_metric_like_patterns,has_copying_risk
0,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
1,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
2,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
3,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
4,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
5,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
6,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
7,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
8,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False
9,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,[],[],False


Validation passed: no obvious reference-specific leakage detected.
Saved validation: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\style_artifact_validation.csv
Saved forbidden terms: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\language_rules\forbidden_reference_terms.json


In [18]:
# ============================================================
# STYLE EXTRACTION AUDIT SUMMARY
# ============================================================

summary_path = STYLE_OUTPUT_DIR / "style_extraction_audit_summary.md"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("# Style Extraction Audit Summary\n\n")

    f.write("## Source\n\n")
    f.write(f"- Reference report: `{REFERENCE_REPORT_PDF}`\n")
    f.write("- Purpose: extract reusable style, structure, layout and formatting patterns only.\n")
    f.write("- The reference report must not be used as a factual source during report generation.\n\n")

    f.write("## Section ranges used\n\n")
    for section, r in section_ranges.items():
        f.write(f"- {section}: pages {r['start_page']}–{r['end_page']}\n")

    f.write("\n## Artifacts created\n\n")
    f.write(f"- `{global_style_path}`\n")
    f.write(f"- `{table_patterns_path}`\n")
    f.write(f"- `{no_copying_path}`\n")
    f.write(f"- `{SECTION_STYLE_DIR}`\n")
    f.write(f"- `{SECTION_BLUEPRINT_DIR}`\n")
    f.write(f"- `{validation_path}`\n\n")

    f.write("## Validation\n\n")
    f.write(f"- Files checked: {len(validation_df)}\n")
    f.write(f"- Files with possible copying/content leakage risk: {len(risky)}\n\n")

    if len(risky):
        f.write("### Files requiring review\n\n")
        for _, row in risky.iterrows():
            f.write(f"- `{row['file']}`\n")

    f.write("\n## Usage rule\n\n")
    f.write(
        "Generation agents should receive only the extracted style artifacts, "
        "not the original reference report text.\n"
    )

print("Saved:", summary_path)
print(summary_path.read_text(encoding="utf-8"))

Saved: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\style_extraction_audit_summary.md
# Style Extraction Audit Summary

## Source

- Reference report: `C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\emirates_nbd_group_2024_ifrs_s1_s2.pdf`
- Purpose: extract reusable style, structure, layout and formatting patterns only.
- The reference report must not be used as a factual source during report generation.

## Section ranges used

- General Requirements: pages 4–7
- Governance: pages 8–18
- Strategy: pages 19–46
- Risk Management: pages 47–54
- Metrics and Targets: pages 55–63

## Artifacts created

- `C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\global_style_guide.json`
- `C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system\table_patterns\table_patterns.json`
- `C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen